# Milestone 5: LLM Summarization and RAG QA

This notebook builds lightweight LLM summarization and retrieval-augmented question answering for Smart Product Intelligence.

It uses the processed Amazon Reviews splits and keeps execution small with `FAST_MODE = True`.

Milestone 6 is not implemented here.

## Setup

The notebook uses a small Hugging Face generation model when available. If model loading or generation fails locally, it falls back to deterministic template-based outputs so the RAG workflow still runs end to end.

In [ ]:
from __future__ import annotations

import ast
import json
import re
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 120)
pd.set_option("display.max_colwidth", 180)

FAST_MODE = True
RANDOM_SEED = 42

if FAST_MODE:
    MAX_REVIEW_ROWS = 6000
    MAX_PRODUCTS = 80
    MAX_REVIEWS_PER_PRODUCT = 25
    SUMMARY_PRODUCTS = 5
    MAX_RAG_CHUNKS = 3000
else:
    MAX_REVIEW_ROWS = None
    MAX_PRODUCTS = 500
    MAX_REVIEWS_PER_PRODUCT = 60
    SUMMARY_PRODUCTS = 20
    MAX_RAG_CHUNKS = 15000

PRIMARY_MODEL = "google/flan-t5-small"
FALLBACK_MODEL = "facebook/bart-base"
QUESTION = "Is this moisturizer good for sensitive skin?"

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
TRAIN_PATH = PROCESSED_DIR / "train.csv"
VALIDATION_PATH = PROCESSED_DIR / "validation.csv"
TEST_PATH = PROCESSED_DIR / "test.csv"

print(f"Project root: {PROJECT_ROOT}")
print(f"FAST_MODE: {FAST_MODE}")
print(f"Primary generation model: {PRIMARY_MODEL}")

## Load Processed Splits

Load the Milestone 0 processed train, validation, and test CSV files.

In [ ]:
for path in [TRAIN_PATH, VALIDATION_PATH, TEST_PATH]:
    if not path.exists():
        raise FileNotFoundError(f"Missing required file: {path}")

train_raw = pd.read_csv(TRAIN_PATH, low_memory=False)
validation_raw = pd.read_csv(VALIDATION_PATH, low_memory=False)
test_raw = pd.read_csv(TEST_PATH, low_memory=False)

raw_reviews = pd.concat(
    [
        train_raw.assign(split="train"),
        validation_raw.assign(split="validation"),
        test_raw.assign(split="test"),
    ],
    ignore_index=True,
)

if FAST_MODE and MAX_REVIEW_ROWS is not None and len(raw_reviews) > MAX_REVIEW_ROWS:
    raw_reviews = raw_reviews.sample(n=MAX_REVIEW_ROWS, random_state=RANDOM_SEED).reset_index(drop=True)

print(f"Loaded review rows: {len(raw_reviews):,}")
print(raw_reviews.columns.tolist())

## Normalize Review Fields

Detect common product, text, title, feature, description, and rating fields. The output is a compact review table used by both summarization and RAG.

In [ ]:
PRODUCT_CANDIDATES = ["product_id", "parent_asin", "asin", "item_id"]
TEXT_CANDIDATES = ["review_text", "text", "review_body", "body", "content"]
TITLE_CANDIDATES = ["product_title", "title"]
REVIEW_TITLE_CANDIDATES = ["review_title", "summary"]
DESCRIPTION_CANDIDATES = ["product_description", "description"]
FEATURE_CANDIDATES = ["product_features", "features"]
RATING_CANDIDATES = ["review_rating", "rating", "average_rating", "overall", "score"]


def first_existing(df: pd.DataFrame, candidates: list[str]) -> str | None:
    return next((column for column in candidates if column in df.columns), None)


def clean_text(value) -> str:
    if value is None:
        return ""
    if isinstance(value, float) and np.isnan(value):
        return ""
    return re.sub(r"\s+", " ", str(value)).strip()


def flatten_text(value) -> str:
    if value is None:
        return ""
    if isinstance(value, float) and np.isnan(value):
        return ""
    if isinstance(value, dict):
        return " ".join(flatten_text(item) for item in value.values())
    if isinstance(value, (list, tuple, set)):
        return " ".join(flatten_text(item) for item in value)
    text = str(value).strip()
    if text.startswith(("[", "{")):
        for parser in (json.loads, ast.literal_eval):
            try:
                return flatten_text(parser(text))
            except Exception:
                pass
    return clean_text(text)


product_col = first_existing(raw_reviews, PRODUCT_CANDIDATES)
text_col = first_existing(raw_reviews, TEXT_CANDIDATES)
title_col = first_existing(raw_reviews, TITLE_CANDIDATES)
review_title_col = first_existing(raw_reviews, REVIEW_TITLE_CANDIDATES)
description_col = first_existing(raw_reviews, DESCRIPTION_CANDIDATES)
feature_col = first_existing(raw_reviews, FEATURE_CANDIDATES)
rating_col = first_existing(raw_reviews, RATING_CANDIDATES)

if text_col is None:
    raise KeyError(f"No review text column found. Available columns: {raw_reviews.columns.tolist()}")
if rating_col is None:
    raise KeyError(f"No rating column found. Available columns: {raw_reviews.columns.tolist()}")

review_df = pd.DataFrame()
review_df["product_id"] = raw_reviews[product_col].astype(str) if product_col else raw_reviews.index.astype(str)
review_df["title"] = raw_reviews[title_col].map(flatten_text) if title_col else "Unknown product"
review_df["review_text"] = raw_reviews[text_col].map(clean_text)
review_df["review_summary"] = raw_reviews[review_title_col].map(clean_text) if review_title_col else ""
review_df["description"] = raw_reviews[description_col].map(flatten_text) if description_col else ""
review_df["features"] = raw_reviews[feature_col].map(flatten_text) if feature_col else ""
review_df["rating"] = pd.to_numeric(raw_reviews[rating_col], errors="coerce")
review_df = review_df[(review_df["review_text"].str.len() > 0) & review_df["rating"].notna()].copy()

print(f"Prepared review rows: {len(review_df):,}")
display(review_df.head())

# Part A: Review Summarization

Aggregate reviews to the product level and create pros, cons, and a short summary. The notebook compares zero-shot prompting with a lightweight prompt-template adaptation that simulates fine-tuning without expensive local training.

In [ ]:
def combine_unique(values: pd.Series, max_items: int, max_chars: int = 400) -> str:
    parts = []
    seen = set()
    for value in values:
        text = clean_text(value)
        if not text:
            continue
        key = text.lower()
        if key in seen:
            continue
        seen.add(key)
        parts.append(text[:max_chars])
        if len(parts) >= max_items:
            break
    return " ".join(parts)


def first_non_empty(values: pd.Series) -> str:
    for value in values:
        text = clean_text(value)
        if text and text.lower() != "unknown":
            return text
    return "Unknown product"


product_rows = []
for product_id, group in review_df.groupby("product_id", dropna=True):
    group = group.sample(frac=1, random_state=RANDOM_SEED)
    positive = group[group["rating"] >= 4]
    negative = group[group["rating"] < 4]
    all_reviews = group.head(MAX_REVIEWS_PER_PRODUCT)
    product_rows.append(
        {
            "product_id": product_id,
            "title": first_non_empty(group["title"]),
            "average_rating": float(group["rating"].mean()),
            "review_count": int(len(group)),
            "pros": combine_unique(positive["review_text"], max_items=4),
            "cons": combine_unique(negative["review_text"], max_items=4),
            "review_corpus": combine_unique(all_reviews["review_text"], max_items=MAX_REVIEWS_PER_PRODUCT),
            "metadata_text": clean_text(
                " ".join(
                    [
                        first_non_empty(group["title"]),
                        combine_unique(group["description"], max_items=2),
                        combine_unique(group["features"], max_items=3),
                    ]
                )
            ),
        }
    )

product_df = pd.DataFrame(product_rows)
product_df = product_df.sort_values(["review_count", "average_rating"], ascending=False).head(MAX_PRODUCTS).reset_index(drop=True)
product_df["short_summary"] = np.where(
    product_df["average_rating"] >= 4,
    "Customers are generally positive, with recurring praise in the review text.",
    "Customer feedback is mixed, so inspect pros and cons before recommending.",
)

print(f"Product-level rows: {len(product_df):,}")
display(product_df[["product_id", "title", "average_rating", "review_count", "pros", "cons", "short_summary"]].head())

## Load Lightweight Generator

Try `google/flan-t5-small` first, then `facebook/bart-base`. If both fail, use a deterministic template fallback.

In [ ]:
generator = None
generator_name = "template-fallback"

try:
    from transformers import pipeline

    try:
        generator = pipeline("text2text-generation", model=PRIMARY_MODEL)
        generator_name = PRIMARY_MODEL
    except Exception as primary_error:
        print(f"Could not load {PRIMARY_MODEL}: {primary_error}")
        generator = pipeline("text2text-generation", model=FALLBACK_MODEL)
        generator_name = FALLBACK_MODEL
except Exception as load_error:
    print(f"No Hugging Face generator available locally: {load_error}")


def template_generate(prompt: str, max_new_tokens: int = 120) -> str:
    if "Question:" in prompt and "Context:" in prompt:
        return "Based on the retrieved reviews, the safest answer is to rely only on the listed evidence. The evidence suggests mixed customer experiences, so confidence depends on how closely the snippets match the question."
    if "Question:" in prompt:
        return "Without retrieved evidence, this answer is speculative and may invent details not present in the reviews."
    return "Pros: Customers mention useful qualities in positive reviews. Cons: Some reviews mention drawbacks or mixed experiences. Summary: Review evidence should be checked before making a recommendation."


def generate_text(prompt: str, max_new_tokens: int = 120) -> str:
    if generator is None:
        return template_generate(prompt, max_new_tokens=max_new_tokens)
    try:
        output = generator(prompt, max_new_tokens=max_new_tokens, truncation=True)
        first = output[0]
        return first.get("generated_text") or first.get("summary_text") or str(first)
    except Exception as generation_error:
        print(f"Generation failed, using template fallback: {generation_error}")
        return template_generate(prompt, max_new_tokens=max_new_tokens)


print(f"Generation backend: {generator_name}")

## Zero-Shot vs Prompt-Template Adaptation

The adapted prompt acts as a lightweight fine-tuning simulation by enforcing the exact product-intelligence output format.

In [ ]:
summary_rows = []
for _, row in product_df.head(SUMMARY_PRODUCTS).iterrows():
    zero_shot_prompt = f"Summarize these customer reviews for a beauty product. Reviews: {row['review_corpus']}"
    adapted_prompt = (
        "You are a product intelligence assistant. Return exactly three fields: Pros, Cons, Summary. "
        f"Product title: {row['title']}. Average rating: {row['average_rating']:.2f}. "
        f"Positive evidence: {row['pros']} Negative or mixed evidence: {row['cons']} Reviews: {row['review_corpus']}"
    )
    summary_rows.append(
        {
            "product_id": row["product_id"],
            "title": row["title"],
            "extractive_pros": row["pros"],
            "extractive_cons": row["cons"],
            "extractive_short_summary": row["short_summary"],
            "zero_shot_summary": generate_text(zero_shot_prompt, max_new_tokens=120),
            "prompt_template_summary": generate_text(adapted_prompt, max_new_tokens=140),
        }
    )

summary_comparison = pd.DataFrame(summary_rows)
display(summary_comparison)

# Part B: Retrieval-Augmented QA

Build a TF-IDF retrieval index from review chunks, retrieve the top 5 relevant snippets, and feed the snippets into the generator.

In [ ]:
rag_chunks = review_df.copy()
rag_chunks["chunk_text"] = (
    rag_chunks["title"].fillna("").map(clean_text)
    + " "
    + rag_chunks["review_summary"].fillna("").map(clean_text)
    + " "
    + rag_chunks["review_text"].fillna("").map(clean_text)
)
rag_chunks = rag_chunks[rag_chunks["chunk_text"].str.len() > 0].reset_index(drop=True)
if FAST_MODE and len(rag_chunks) > MAX_RAG_CHUNKS:
    rag_chunks = rag_chunks.sample(n=MAX_RAG_CHUNKS, random_state=RANDOM_SEED).reset_index(drop=True)

retriever = TfidfVectorizer(max_features=50000, ngram_range=(1, 2), min_df=1, sublinear_tf=True)
retrieval_matrix = retriever.fit_transform(rag_chunks["chunk_text"])


def retrieve_reviews(question: str, top_k: int = 5) -> pd.DataFrame:
    query_vector = retriever.transform([question])
    scores = cosine_similarity(query_vector, retrieval_matrix).ravel()
    top_indices = np.argsort(scores)[::-1][:top_k]
    results = rag_chunks.iloc[top_indices][["product_id", "title", "rating", "review_text", "chunk_text"]].copy()
    results["similarity_score"] = scores[top_indices]
    return results.reset_index(drop=True)


retrieved = retrieve_reviews(QUESTION, top_k=5)
display(retrieved[["product_id", "title", "rating", "similarity_score", "review_text"]])

## Grounded and Non-Grounded Answers

The grounded answer receives retrieved review evidence. The non-grounded answer receives only the user question.

In [ ]:
def build_context(retrieved_df: pd.DataFrame) -> str:
    lines = []
    for idx, row in retrieved_df.iterrows():
        lines.append(
            f"Evidence {idx + 1}: title={row['title']}; rating={row['rating']}; review={row['review_text'][:450]}"
        )
    return "\n".join(lines)


context = build_context(retrieved)
grounded_prompt = (
    "Answer the user question using only the retrieved review context. "
    "If the evidence is weak, say that.\n"
    f"Question: {QUESTION}\nContext:\n{context}\nAnswer:"
)
nongrounded_prompt = f"Question: {QUESTION}\nAnswer as a helpful beauty product assistant:"

grounded_answer = generate_text(grounded_prompt, max_new_tokens=140)
nongrounded_answer = generate_text(nongrounded_prompt, max_new_tokens=120)
confidence_score = float(np.clip(retrieved["similarity_score"].head(5).mean() * 3, 0, 1))

rag_result = {
    "question": QUESTION,
    "grounded_answer": grounded_answer,
    "non_grounded_answer": nongrounded_answer,
    "confidence_score": confidence_score,
    "retrieved_evidence_snippets": retrieved["review_text"].head(5).tolist(),
}

print("Answer:")
print(rag_result["grounded_answer"])
print("Confidence score:", round(rag_result["confidence_score"], 3))
print("Evidence snippets:")
for snippet in rag_result["retrieved_evidence_snippets"]:
    print("-", snippet[:300])

display(pd.DataFrame([rag_result]))

## Hallucination Analysis

Ungrounded answers can invent unsupported facts. This section asks questions where review evidence may be weak and flags risky non-grounded answers.

In [ ]:
risk_questions = [
    "Does this product contain SPF 50?",
    "Is this product fragrance free?",
    "Is this safe to use during pregnancy?",
]


def evidence_has_keywords(question: str, evidence_text: str) -> bool:
    terms = [term for term in re.findall(r"[a-zA-Z]+", question.lower()) if len(term) >= 4]
    evidence_lower = evidence_text.lower()
    return any(term in evidence_lower for term in terms)


hallucination_rows = []
for question in risk_questions:
    evidence = retrieve_reviews(question, top_k=5)
    evidence_text = " ".join(evidence["review_text"].tolist())
    grounded = generate_text(
        f"Answer only from this context. Question: {question}\nContext:\n{build_context(evidence)}\nAnswer:",
        max_new_tokens=120,
    )
    ungrounded = generate_text(f"Question: {question}\nAnswer confidently:", max_new_tokens=100)
    has_support = evidence_has_keywords(question, evidence_text)
    risky_words = ["yes", "definitely", "contains", "safe", "guaranteed", "fragrance free", "spf"]
    risk = (not has_support) and any(word in ungrounded.lower() for word in risky_words)
    hallucination_rows.append(
        {
            "question": question,
            "grounded_answer": grounded,
            "ungrounded_answer": ungrounded,
            "evidence_has_question_terms": has_support,
            "hallucination_risk": risk,
            "top_evidence": evidence["review_text"].iloc[0] if len(evidence) else "",
        }
    )

hallucination_df = pd.DataFrame(hallucination_rows)
display(hallucination_df)

## Complete

Milestone 5 is complete. Milestone 6 is not implemented.

In [ ]:
print("Milestone 5 Complete")